# Phase 4: Data Preparation

## Objective

The objective of this notebook is to transform the raw NexaCart marketplace datasets into clean, consistent, and analysis-ready datasets.

Data preparation consists of two major stages:

1. **Data Cleaning**
   - Correct structural issues
   - Handle missing values
   - Validate data types
   - Remove invalid records
   - Standardize values
   - Validate business rules

2. **Feature Engineering**
   - Create business-oriented features
   - Generate analytical metrics
   - Prepare datasets for EDA, SQL analysis, and Power BI

The cleaned datasets generated in this notebook will serve as the single source of truth for all downstream analysis.

In [1]:
# ==========================================================
# Phase 4 : Data Preparation
# Import Required Libraries
# ==========================================================

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd

# Display Settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.float_format", "{:.2f}".format)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# ==========================================================
# Project Paths
# ==========================================================

PROJECT_ROOT = Path.cwd().parent

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw"

CLEAN_DATA_PATH = PROJECT_ROOT / "data" / "cleaned"

EXPORT_PATH = PROJECT_ROOT / "data" / "exports"

RAW_FILE = RAW_DATA_PATH / "NexaCart_Data.xlsx"

# Create folders if they don't exist
CLEAN_DATA_PATH.mkdir(parents=True, exist_ok=True)
EXPORT_PATH.mkdir(parents=True, exist_ok=True)

print("Project paths configured.")

Project paths configured.


In [3]:
# ==========================================================
# Verify Dataset
# ==========================================================

if not RAW_FILE.exists():
    raise FileNotFoundError(f"{RAW_FILE} not found.")

print("Raw dataset located successfully.")

Raw dataset located successfully.


In [4]:
# ==========================================================
# Load Workbook
# ==========================================================

excel = pd.ExcelFile(RAW_FILE)

sheet_names = excel.sheet_names

print(f"Workbook loaded successfully.")
print(f"Total Sheets : {len(sheet_names)}")

Workbook loaded successfully.
Total Sheets : 9


In [5]:
# ==========================================================
# Load All Worksheets
# ==========================================================

datasets = {
    sheet: pd.read_excel(RAW_FILE, sheet_name=sheet)
    for sheet in sheet_names
}

print(f"Loaded {len(datasets)} datasets.")

Loaded 9 datasets.


In [6]:
# ==========================================================
# Create Working Copies
# ==========================================================

cleaned_datasets = {
    name: df.copy(deep=True)
    for name, df in datasets.items()
}

print(f"Created working copies for {len(cleaned_datasets)} datasets.")

Created working copies for 9 datasets.


In [7]:
# ==========================================================
# Data Cleaning Checklist
# ==========================================================

cleaning_checklist = pd.DataFrame({

    "Dataset": [
        "product_category_name_translation",
        "products_dataset",
        "products_dataset",
        "products_dataset",
        "orders_dataset",
        "orders_dataset",
        "geolocation_dataset",
        "order_reviews_dataset",
        "customers_dataset"
    ],

    "Issue": [
        "Incorrect column names",
        "Missing product category",
        "Missing product dimensions",
        "Missing product weight",
        "Datetime validation",
        "Order status validation",
        "Duplicate ZIP mappings",
        "Review score validation",
        "Customer uniqueness validation"
    ],

    "Priority": [
        "High",
        "High",
        "Medium",
        "Medium",
        "High",
        "Medium",
        "Low",
        "High",
        "Medium"
    ],

    "Planned Action": [
        "Rename columns",
        "Analyze missing values",
        "Analyze missing values",
        "Analyze missing values",
        "Validate timestamps",
        "Validate categories",
        "Retain for analysis",
        "Validate score range",
        "Validate unique IDs"
    ],

    "Status": [
        "Pending"
    ] * 9
})

cleaning_checklist

,Dataset,Issue,Priority,Planned Action,Status
0,product_category_name_translation,Incorrect column names,High,Rename columns,Pending
1,products_dataset,Missing product category,High,Analyze missing values,Pending
2,products_dataset,Missing product dimensions,Medium,Analyze missing values,Pending
3,products_dataset,Missing product weight,Medium,Analyze missing values,Pending
4,orders_dataset,Datetime validation,High,Validate timestamps,Pending
5,orders_dataset,Order status validation,Medium,Validate categories,Pending
6,geolocation_dataset,Duplicate ZIP mappings,Low,Retain for analysis,Pending
7,order_reviews_dataset,Review score validation,High,Validate score range,Pending
8,customers_dataset,Customer uniqueness validation,Medium,Validate unique IDs,Pending


# Header Validation

The objective of this section is to validate the structure of every dataset before performing any cleaning operations.

The validation checks:

- Duplicate column names
- Empty column names
- Generic column names (e.g., Column1, Column2)
- Naming consistency

This helps identify structural issues that may affect downstream processing.

In [8]:
# ==========================================================
# Header Validation
# ==========================================================

def validate_headers(dataset_name: str, df: pd.DataFrame):

    issues = []

    # Duplicate column names
    duplicate_columns = df.columns[df.columns.duplicated()].tolist()

    if duplicate_columns:
        issues.append(
            f"Duplicate columns: {duplicate_columns}"
        )

    # Empty column names
    empty_columns = [
        col
        for col in df.columns
        if str(col).strip() == ""
    ]

    if empty_columns:
        issues.append(
            "Empty column names found"
        )

    # Generic column names
    generic_columns = [
        col
        for col in df.columns
        if str(col).lower().startswith("column")
    ]

    if generic_columns:
        issues.append(
            f"Generic column names: {generic_columns}"
        )

    return {
        "Dataset": dataset_name,
        "Status": "Pass" if len(issues) == 0 else "Fail",
        "Issues": "; ".join(issues) if issues else "No issues detected"
    }

In [9]:
header_validation = []

for name, df in cleaned_datasets.items():

    header_validation.append(
        validate_headers(name, df)
    )

header_validation = pd.DataFrame(header_validation)

header_validation

,Dataset,Status,Issues
0,sellers_dataset,Pass,No issues detected
1,product_category_name_translati,Fail,"Generic column names: ['Column1', 'Column2']"
2,products_dataset,Pass,No issues detected
3,order_reviews_dataset,Pass,No issues detected
4,orders_dataset,Pass,No issues detected
5,order_payments_dataset,Pass,No issues detected
6,order_items_dataset,Pass,No issues detected
7,geolocation_dataset,Pass,No issues detected
8,customers_dataset,Pass,No issues detected


In [10]:
if (
    header_validation["Status"] == "Fail"
).any():

    cleaning_checklist.loc[
        cleaning_checklist["Issue"] == "Incorrect column names",
        "Status"
    ] = "In Progress"

cleaning_checklist

,Dataset,Issue,Priority,Planned Action,Status
0,product_category_name_translation,Incorrect column names,High,Rename columns,In Progress
1,products_dataset,Missing product category,High,Analyze missing values,Pending
2,products_dataset,Missing product dimensions,Medium,Analyze missing values,Pending
3,products_dataset,Missing product weight,Medium,Analyze missing values,Pending
4,orders_dataset,Datetime validation,High,Validate timestamps,Pending
5,orders_dataset,Order status validation,Medium,Validate categories,Pending
6,geolocation_dataset,Duplicate ZIP mappings,Low,Retain for analysis,Pending
7,order_reviews_dataset,Review score validation,High,Validate score range,Pending
8,customers_dataset,Customer uniqueness validation,Medium,Validate unique IDs,Pending


In [11]:
translation_df = cleaned_datasets[
    "product_category_name_translati"
]

translation_df.head()

,Column1,Column2
0,product_category_name,product_category_name_english
1,beleza_saude,health_beauty
2,informatica_acessorios,computers_accessories
3,automotivo,auto
4,cama_mesa_banho,bed_bath_table


## Header Correction

The product category translation dataset was found to contain generic column names (`Column1` and `Column2`).

Further inspection showed that the first row actually contains the correct column names.

To preserve the integrity of the dataset, the first row is promoted to the header, after which it is removed from the dataset.

In [12]:
# ==========================================================
# Correct Translation Dataset Headers
# ==========================================================

translation_key = "product_category_name_translati"

translation_df = cleaned_datasets[translation_key].copy()

# Promote first row to header
translation_df.columns = translation_df.iloc[0]

# Remove the first row
translation_df = translation_df.iloc[1:].reset_index(drop=True)

# Replace the cleaned dataframe
cleaned_datasets[translation_key] = translation_df

print("Header correction completed successfully.")

Header correction completed successfully.


In [13]:
translation_df.head()

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [14]:
cleaning_checklist.loc[
    cleaning_checklist["Issue"] == "Incorrect column names",
    "Status"
] = "Completed"

cleaning_checklist

,Dataset,Issue,Priority,Planned Action,Status
0,product_category_name_translation,Incorrect column names,High,Rename columns,Completed
1,products_dataset,Missing product category,High,Analyze missing values,Pending
2,products_dataset,Missing product dimensions,Medium,Analyze missing values,Pending
3,products_dataset,Missing product weight,Medium,Analyze missing values,Pending
4,orders_dataset,Datetime validation,High,Validate timestamps,Pending
5,orders_dataset,Order status validation,Medium,Validate categories,Pending
6,geolocation_dataset,Duplicate ZIP mappings,Low,Retain for analysis,Pending
7,order_reviews_dataset,Review score validation,High,Validate score range,Pending
8,customers_dataset,Customer uniqueness validation,Medium,Validate unique IDs,Pending


In [15]:
header_validation = []

for name, df in cleaned_datasets.items():
    header_validation.append(
        validate_headers(name, df)
    )

header_validation = pd.DataFrame(header_validation)

header_validation

,Dataset,Status,Issues
0,sellers_dataset,Pass,No issues detected
1,product_category_name_translati,Pass,No issues detected
2,products_dataset,Pass,No issues detected
3,order_reviews_dataset,Pass,No issues detected
4,orders_dataset,Pass,No issues detected
5,order_payments_dataset,Pass,No issues detected
6,order_items_dataset,Pass,No issues detected
7,geolocation_dataset,Pass,No issues detected
8,customers_dataset,Pass,No issues detected


# Products Dataset

## Objective

The Products dataset contains product-level information, including category, dimensions, weight, and physical characteristics.

This dataset will be validated and cleaned before being used in downstream business analysis.

The following quality checks will be performed:

- Dataset overview
- Data type validation
- Missing value analysis
- Duplicate validation
- Business rule validation
- Cleaning decisions
- Post-cleaning validation

In [16]:
# ==========================================================
# Products Dataset
# ==========================================================

products_df = cleaned_datasets["products_dataset"].copy()

print(f"Rows    : {products_df.shape[0]:,}")
print(f"Columns : {products_df.shape[1]}")

products_df.head()

Rows    : 32,951
Columns : 9


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.00,287.00,1.00,225.00,16.00,10.00,14.00
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.00,276.00,1.00,1000.00,30.00,18.00,20.00
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.00,250.00,1.00,154.00,18.00,9.00,15.00
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.00,261.00,1.00,371.00,26.00,4.00,26.00
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.00,402.00,4.00,625.00,20.00,17.00,13.00


In [17]:
# ==========================================================
# Products Dataset Summary
# ==========================================================

products_summary = pd.DataFrame({

    "Data Type": products_df.dtypes,

    "Missing Values": products_df.isna().sum(),

    "Missing %": (
        products_df.isna().mean()*100
    ).round(2),

    "Unique Values": products_df.nunique()

})

products_summary

,Data Type,Missing Values,Missing %,Unique Values
product_id,str,0,0.00,32951
product_category_name,str,610,1.85,73
product_name_lenght,float64,610,1.85,66
product_description_lenght,float64,610,1.85,2960
product_photos_qty,float64,610,1.85,19
product_weight_g,float64,2,0.01,2204
product_length_cm,float64,2,0.01,99
product_height_cm,float64,2,0.01,102
product_width_cm,float64,2,0.01,95


In [19]:
missing_percentage = (
    products_df
    .isna()
    .mean()
    *100
).round(2)

missing_percentage.sort_values(
    ascending=False
)

product_category_name        1.85
product_name_lenght          1.85
product_description_lenght   1.85
product_photos_qty           1.85
product_weight_g             0.01
product_length_cm            0.01
product_height_cm            0.01
product_width_cm             0.01
product_id                   0.00
dtype: float64

In [20]:
duplicates = products_df.duplicated().sum()

print(f"Duplicate Rows : {duplicates}")

Duplicate Rows : 0


In [21]:
duplicate_products = (
    products_df["product_id"]
    .duplicated()
    .sum()
)

print(
    f"Duplicate Product IDs : {duplicate_products}"
)

Duplicate Product IDs : 0


In [22]:
invalid_weight = products_df[
    products_df["product_weight_g"] <= 0
]

print(
    f"Invalid Weights : {len(invalid_weight)}"
)

Invalid Weights : 4


In [23]:
invalid_length = products_df[
    products_df["product_length_cm"] <=0
]

print(
    len(invalid_length)
)

0


In [24]:
products_df[
    "product_category_name"
].value_counts(dropna=False).head()

product_category_name
cama_mesa_banho          3029
esporte_lazer            2867
moveis_decoracao         2657
beleza_saude             2444
utilidades_domesticas    2335
Name: count, dtype: int64

In [25]:
products_cleaning = pd.DataFrame({

    "Issue":[

        "Missing Category",

        "Missing Weight",

        "Missing Length",

        "Missing Height",

        "Missing Width",

        "Duplicate Product IDs"

    ],

    "Observation":[

        "",

        "",

        "",

        "",

        "",

        ""

    ],

    "Decision":[

        "",

        "",

        "",

        "",

        "",

        ""

    ]

})

products_cleaning

,Issue,Observation,Decision
0,Missing Category,,
1,Missing Weight,,
2,Missing Length,,
3,Missing Height,,
4,Missing Width,,
5,Duplicate Product IDs,,


## Missing Product Metadata Analysis

The products dataset contains missing values across several metadata-related columns.

Before applying any treatment, the affected records are inspected to determine whether the missing values occur independently or belong to the same products.

This investigation helps determine the most appropriate cleaning strategy.

In [26]:
# ==========================================================
# Products with Missing Metadata
# ==========================================================

missing_products = products_df[
    products_df["product_category_name"].isna()
]

print(f"Products with missing metadata: {len(missing_products)}")

missing_products.head()

Products with missing metadata: 610


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.00,17.00,14.00,12.00
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.00,16.00,7.00,20.00
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.00,20.00,20.00,20.00
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.00,41.00,30.00,41.00
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.00,35.00,7.00,12.00


In [27]:
missing_products.isna().mean().mul(100).round(2)

product_id                     0.00
product_category_name        100.00
product_name_lenght          100.00
product_description_lenght   100.00
product_photos_qty           100.00
product_weight_g               0.16
product_length_cm              0.16
product_height_cm              0.16
product_width_cm               0.16
dtype: float64

In [28]:
# ==========================================================
# Invalid Product Weights
# ==========================================================

invalid_weight_products = products_df[
    products_df["product_weight_g"] <= 0
]

invalid_weight_products

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
9769,81781c0fed9fe1ad6e8c81fca1e1cb08,cama_mesa_banho,51.00,529.00,1.00,0.00,30.00,25.00,30.00
13683,8038040ee2a71048d4bdbbdc985b69ab,cama_mesa_banho,48.00,528.00,1.00,0.00,30.00,25.00,30.00
14997,36ba42dd187055e1fbe943b2d11430ca,cama_mesa_banho,53.00,528.00,1.00,0.00,30.00,25.00,30.00
32079,e673e90efa65a5409ff4196c038bb5af,cama_mesa_banho,53.00,528.00,1.00,0.00,30.00,25.00,30.00


In [29]:
dimension_columns = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

products_df[
    products_df[dimension_columns]
    .isna()
    .any(axis=1)
]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,bebes,60.00,865.00,3.00,NaN,NaN,NaN,NaN
18851,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
products_cleaning = pd.DataFrame({

    "Issue":[
        "Missing Category",
        "Missing Weight",
        "Missing Length",
        "Missing Height",
        "Missing Width",
        "Duplicate Product IDs",
        "Invalid Weight"
    ],

    "Observation":[
        "610 rows (1.85%) missing product metadata",
        "2 rows missing",
        "2 rows missing",
        "2 rows missing",
        "2 rows missing",
        "No duplicates found",
        "4 products have non-positive weight"
    ],

    "Decision":[
        "",
        "",
        "",
        "",
        "",
        "No action required",
        ""
    ]
})

products_cleaning

,Issue,Observation,Decision
0,Missing Category,610 rows (1.85%) missing product metadata,
1,Missing Weight,2 rows missing,
2,Missing Length,2 rows missing,
3,Missing Height,2 rows missing,
4,Missing Width,2 rows missing,
5,Duplicate Product IDs,No duplicates found,No action required
6,Invalid Weight,4 products have non-positive weight,


## Cleaning Decisions

The Products dataset was assessed using data quality validation techniques before any transformations were applied.

### Missing Product Metadata

A total of 610 products (1.85%) were found to have missing product metadata including:

- Product Category
- Product Name Length
- Product Description Length
- Product Photos Quantity

These missing values occur together, indicating incomplete catalog information rather than isolated missing fields.

Since no reliable method exists to reconstruct this information, these values will be preserved as missing.

---

### Missing Product Dimensions

Only two products contain missing physical dimensions.

Because the missing rate is extremely low (0.01%), these records will be retained for now and handled during feature engineering if required.

---

### Invalid Product Weight

Four products contain a recorded weight of zero grams.

Since a physical product cannot realistically have zero weight, these values are considered invalid.

The values will be converted to missing (NaN) rather than replaced with arbitrary estimates.

---

### Duplicate Products

No duplicate Product IDs were identified.

No action is required.

In [31]:
# ==========================================================
# Replace Invalid Product Weights
# ==========================================================

products_df.loc[
    products_df["product_weight_g"] <= 0,
    "product_weight_g"
] = np.nan

print("Invalid weights converted to missing values.")

Invalid weights converted to missing values.


In [32]:
(products_df["product_weight_g"] <= 0).sum()

np.int64(0)

In [33]:
cleaned_datasets["products_dataset"] = products_df

In [34]:
cleaning_checklist.loc[
    cleaning_checklist["Issue"] == "Missing product weight",
    "Status"
] = "Completed"

In [35]:
products_df.isna().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                6
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

## Post-Cleaning Validation

After applying the cleaning transformations, the Products dataset is validated to ensure that:

- Invalid values have been corrected.
- Duplicate records are absent.
- Missing values reflect genuine unknown information.
- The dataset is ready for downstream analysis.

In [36]:
# ==========================================================
# Post-Cleaning Validation
# ==========================================================

products_validation = pd.DataFrame({

    "Validation Check": [

        "Duplicate Rows",

        "Duplicate Product IDs",

        "Invalid Weight Values",

        "Missing Product Metadata",

        "Missing Physical Dimensions"

    ],

    "Result": [

        products_df.duplicated().sum(),

        products_df["product_id"].duplicated().sum(),

        (products_df["product_weight_g"] <= 0).sum(),

        products_df["product_category_name"].isna().sum(),

        products_df[
            [
                "product_length_cm",
                "product_height_cm",
                "product_width_cm"
            ]
        ].isna().any(axis=1).sum()

    ],

    "Status": [

        "Pass",

        "Pass",

        "Pass",

        "Accepted",

        "Accepted"

    ]

})

products_validation

,Validation Check,Result,Status
0,Duplicate Rows,0,Pass
1,Duplicate Product IDs,0,Pass
2,Invalid Weight Values,0,Pass
3,Missing Product Metadata,610,Accepted
4,Missing Physical Dimensions,2,Accepted


In [37]:
# ==========================================================
# Update Cleaning Checklist
# ==========================================================

completed_issues = [
    "Missing product category",
    "Missing product dimensions",
    "Missing product weight"
]

cleaning_checklist.loc[
    cleaning_checklist["Issue"].isin(completed_issues),
    "Status"
] = "Completed"

cleaning_checklist

,Dataset,Issue,Priority,Planned Action,Status
0,product_category_name_translation,Incorrect column names,High,Rename columns,Completed
1,products_dataset,Missing product category,High,Analyze missing values,Completed
2,products_dataset,Missing product dimensions,Medium,Analyze missing values,Completed
3,products_dataset,Missing product weight,Medium,Analyze missing values,Completed
4,orders_dataset,Datetime validation,High,Validate timestamps,Pending
5,orders_dataset,Order status validation,Medium,Validate categories,Pending
6,geolocation_dataset,Duplicate ZIP mappings,Low,Retain for analysis,Pending
7,order_reviews_dataset,Review score validation,High,Validate score range,Pending
8,customers_dataset,Customer uniqueness validation,Medium,Validate unique IDs,Pending


## Products Dataset Summary

### Cleaning Activities Performed

- Validated dataset structure.
- Verified primary key uniqueness.
- Confirmed absence of duplicate records.
- Investigated missing product metadata.
- Investigated missing physical dimensions.
- Converted invalid product weights (0 grams) to missing values.
- Preserved genuine missing values where reliable imputation was not possible.

### Outcome

The Products dataset is structurally valid and suitable for downstream analysis.

Remaining missing values represent genuine unknown information and will be retained to avoid introducing artificial bias into subsequent analyses.

# Orders Dataset

## Objective

The Orders dataset represents the complete lifecycle of customer orders, from purchase through delivery.

Since this dataset serves as the central transactional table within the marketplace, its quality directly impacts all downstream analyses.

The following validation activities will be performed:

- Dataset overview
- Primary key validation
- Duplicate validation
- Missing value analysis
- Datetime validation
- Order status validation
- Business rule validation
- Cleaning decisions
- Post-cleaning validation

In [38]:
# ==========================================================
# Orders Dataset
# ==========================================================

orders_df = cleaned_datasets["orders_dataset"].copy()

print(f"Rows    : {orders_df.shape[0]:,}")
print(f"Columns : {orders_df.shape[1]}")

orders_df.head()

Rows    : 99,441
Columns : 8


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


In [39]:
# ==========================================================
# Orders Dataset Summary
# ==========================================================

orders_summary = pd.DataFrame({

    "Data Type": orders_df.dtypes,

    "Missing Values": orders_df.isna().sum(),

    "Missing %": (
        orders_df.isna().mean()*100
    ).round(2),

    "Unique Values": orders_df.nunique()

})

orders_summary

,Data Type,Missing Values,Missing %,Unique Values
order_id,str,0,0.00,99441
customer_id,str,0,0.00,99441
order_status,str,0,0.00,8
order_purchase_timestamp,datetime64[us],0,0.00,98875
order_approved_at,datetime64[us],160,0.16,90733
order_delivered_carrier_date,datetime64[us],1783,1.79,81018
order_delivered_customer_date,datetime64[us],2965,2.98,95664
order_estimated_delivery_date,datetime64[us],0,0.00,459


In [40]:
duplicate_rows = orders_df.duplicated().sum()

print(f"Duplicate Rows : {duplicate_rows}")

Duplicate Rows : 0


In [41]:
duplicate_order_ids = (
    orders_df["order_id"]
    .duplicated()
    .sum()
)

print(
    f"Duplicate Order IDs : {duplicate_order_ids}"
)

Duplicate Order IDs : 0


In [42]:
missing_percentage = (
    orders_df
    .isna()
    .mean()
    .mul(100)
    .round(2)
)

missing_percentage.sort_values(
    ascending=False
)

order_delivered_customer_date   2.98
order_delivered_carrier_date    1.79
order_approved_at               0.16
order_id                        0.00
customer_id                     0.00
order_status                    0.00
order_purchase_timestamp        0.00
order_estimated_delivery_date   0.00
dtype: float64

In [43]:
orders_df["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [44]:
orders_df.dtypes

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [45]:
invalid_delivery = orders_df[
    orders_df["order_delivered_customer_date"] <
    orders_df["order_purchase_timestamp"]
]

print(len(invalid_delivery))

0


In [46]:
invalid_approval = orders_df[
    orders_df["order_approved_at"] <
    orders_df["order_purchase_timestamp"]
]

print(len(invalid_approval))

0


In [47]:
invalid_estimate = orders_df[
    orders_df["order_estimated_delivery_date"] <
    orders_df["order_purchase_timestamp"]
]

print(len(invalid_estimate))

0


In [48]:
orders_cleaning = pd.DataFrame({

    "Issue":[
        "Duplicate Orders",
        "Duplicate Order IDs",
        "Missing Datetimes",
        "Invalid Order Status",
        "Delivery Before Purchase",
        "Approval Before Purchase",
        "Estimated Delivery Before Purchase"
    ],

    "Observation":[
        "",
        "",
        "",
        "",
        "",
        "",
        ""
    ],

    "Decision":[
        "",
        "",
        "",
        "",
        "",
        "",
        ""
    ]

})

orders_cleaning

,Issue,Observation,Decision
0,Duplicate Orders,,
1,Duplicate Order IDs,,
2,Missing Datetimes,,
3,Invalid Order Status,,
4,Delivery Before Purchase,,
5,Approval Before Purchase,,
6,Estimated Delivery Before Purchase,,


## Missing Datetime Investigation

Missing datetime values may represent valid stages within the order lifecycle rather than data quality issues.

Before applying any treatment, the missing values are analyzed against the order status to determine whether they are expected from a business perspective.

In [49]:
# ==========================================================
# Missing Approval Timestamp
# ==========================================================

orders_df[
    orders_df["order_approved_at"].isna()
]["order_status"].value_counts()

order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64

In [50]:
# ==========================================================
# Missing Carrier Timestamp
# ==========================================================

orders_df[
    orders_df["order_delivered_carrier_date"].isna()
]["order_status"].value_counts()

order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64

In [51]:
# ==========================================================
# Missing Customer Delivery Timestamp
# ==========================================================

orders_df[
    orders_df["order_delivered_customer_date"].isna()
]["order_status"].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

In [52]:
# ==========================================================
# Delivered Orders Missing Approval Timestamp
# ==========================================================

delivered_missing_approval = orders_df[
    (orders_df["order_status"] == "delivered") &
    (orders_df["order_approved_at"].isna())
]

delivered_missing_approval

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
5323,e04abd8149ef81b95221e88f6ed9ab6a,2127dc6603ac33544953ef05ec155771,delivered,2017-02-18 14:40:00,NaT,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17
16567,8a9adc69528e1001fc68dd0aaebbb54a,4c1ccc74e00993733742a3c786dc3c1f,delivered,2017-02-18 12:45:31,NaT,2017-02-23 09:01:52,2017-03-02 10:05:06,2017-03-21
19031,7013bcfc1c97fe719a7b5e05e61c12db,2941af76d38100e0f8740a374f1a5dc3,delivered,2017-02-18 13:29:47,NaT,2017-02-22 16:25:25,2017-03-01 08:07:38,2017-03-17
22663,5cf925b116421afa85ee25e99b4c34fb,29c35fc91fc13fb5073c8f30505d860d,delivered,2017-02-18 16:48:35,NaT,2017-02-22 11:23:10,2017-03-09 07:28:47,2017-03-31
23156,12a95a3c06dbaec84bcfb0e2da5d228a,1e101e0daffaddce8159d25a8e53f2b2,delivered,2017-02-17 13:05:55,NaT,2017-02-22 11:23:11,2017-03-02 11:09:19,2017-03-20
26800,c1d4211b3dae76144deccd6c74144a88,684cb238dc5b5d6366244e0e0776b450,delivered,2017-01-19 12:48:08,NaT,2017-01-25 14:56:50,2017-01-30 18:16:01,2017-03-01
38290,d69e5d356402adc8cf17e08b5033acfb,68d081753ad4fe22fc4d410a9eb1ca01,delivered,2017-02-19 01:28:47,NaT,2017-02-23 03:11:48,2017-03-02 03:41:58,2017-03-27
39334,d77031d6a3c8a52f019764e68f211c69,0bf35cac6cc7327065da879e2d90fae8,delivered,2017-02-18 11:04:19,NaT,2017-02-23 07:23:36,2017-03-02 16:15:23,2017-03-22
48401,7002a78c79c519ac54022d4f8a65e6e8,d5de688c321096d15508faae67a27051,delivered,2017-01-19 22:26:59,NaT,2017-01-27 11:08:05,2017-02-06 14:22:19,2017-03-16
61743,2eecb0d85f281280f79fa00f9cec1a95,a3d3c38e58b9d2dfb9207cab690b6310,delivered,2017-02-17 17:21:55,NaT,2017-02-22 11:42:51,2017-03-03 12:16:03,2017-03-20


In [53]:
# ==========================================================
# Delivered Orders Missing Carrier Timestamp
# ==========================================================

delivered_missing_carrier = orders_df[
    (orders_df["order_status"] == "delivered") &
    (orders_df["order_delivered_carrier_date"].isna())
]

delivered_missing_carrier

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
73222,2aa91108853cecb43c84a5dc5b277475,afeb16c7f46396c0ed54acb45ccaaa40,delivered,2017-09-29 08:52:58,2017-09-29 09:07:16,NaT,2017-11-20 19:44:47,2017-11-14
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23


In [54]:
# ==========================================================
# Delivered Orders Missing Customer Delivery Timestamp
# ==========================================================

delivered_missing_customer = orders_df[
    (orders_df["order_status"] == "delivered") &
    (orders_df["order_delivered_customer_date"].isna())
]

delivered_missing_customer

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaT,2018-07-30
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaT,2018-07-30
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaT,2018-07-24
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaT,2018-06-26
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaT,2018-07-19


## Cleaning Decisions

The Orders dataset passed all structural validation checks.

### Duplicate Records

No duplicate rows or duplicate Order IDs were identified.

No action was required.

---

### Missing Datetime Values

Missing datetime values were investigated against the order lifecycle.

The majority of missing timestamps correspond to legitimate business states such as:

- Created
- Processing
- Invoiced
- Cancelled
- Unavailable

These records represent incomplete order lifecycles rather than data quality issues.

---

### Delivered Order Anomalies

A very small number of delivered orders contained missing lifecycle timestamps:

- 14 orders missing approval timestamp
- 2 orders missing carrier timestamp
- 8 orders missing customer delivery timestamp

These represent only 0.024% of the dataset.

Because no reliable method exists to reconstruct the missing timestamps, the records are retained and documented as data quality anomalies.

---

### Outcome

No records were removed.

No timestamps were imputed.

The dataset retains its original business meaning while preserving transparency regarding the identified anomalies.

In [55]:
orders_cleaning.loc[:, "Observation"] = [

    "No duplicate rows",

    "No duplicate Order IDs",

    "Missing timestamps mostly correspond to valid business states",

    "All statuses are valid",

    "No invalid records",

    "No invalid records",

    "No invalid records"

]

orders_cleaning.loc[:, "Decision"] = [

    "No action required",

    "No action required",

    "Retain missing values and document anomalies",

    "No action required",

    "No action required",

    "No action required",

    "No action required"

]

orders_cleaning

,Issue,Observation,Decision
0,Duplicate Orders,No duplicate rows,No action required
1,Duplicate Order IDs,No duplicate Order IDs,No action required
2,Missing Datetimes,Missing timestamps mostly correspond to valid business states,Retain missing values and document anomalies
3,Invalid Order Status,All statuses are valid,No action required
4,Delivery Before Purchase,No invalid records,No action required
5,Approval Before Purchase,No invalid records,No action required
6,Estimated Delivery Before Purchase,No invalid records,No action required


In [56]:
# ==========================================================
# Orders Dataset Validation Summary
# ==========================================================

orders_validation = pd.DataFrame({

    "Validation Check":[

        "Duplicate Rows",

        "Duplicate Order IDs",

        "Invalid Order Status",

        "Invalid Timeline",

        "Missing Datetime Values"

    ],

    "Result":[

        duplicate_rows,

        duplicate_order_ids,

        0,

        0,

        "Accepted"

    ],

    "Status":[

        "Pass",

        "Pass",

        "Pass",

        "Pass",

        "Accepted"

    ]

})

orders_validation

,Validation Check,Result,Status
0,Duplicate Rows,0,Pass
1,Duplicate Order IDs,0,Pass
2,Invalid Order Status,0,Pass
3,Invalid Timeline,0,Pass
4,Missing Datetime Values,Accepted,Accepted


In [57]:
completed_issues = [
    "Datetime validation",
    "Order status validation"
]

cleaning_checklist.loc[
    cleaning_checklist["Issue"].isin(completed_issues),
    "Status"
] = "Completed"

cleaning_checklist

,Dataset,Issue,Priority,Planned Action,Status
0,product_category_name_translation,Incorrect column names,High,Rename columns,Completed
1,products_dataset,Missing product category,High,Analyze missing values,Completed
2,products_dataset,Missing product dimensions,Medium,Analyze missing values,Completed
3,products_dataset,Missing product weight,Medium,Analyze missing values,Completed
4,orders_dataset,Datetime validation,High,Validate timestamps,Completed
5,orders_dataset,Order status validation,Medium,Validate categories,Completed
6,geolocation_dataset,Duplicate ZIP mappings,Low,Retain for analysis,Pending
7,order_reviews_dataset,Review score validation,High,Validate score range,Pending
8,customers_dataset,Customer uniqueness validation,Medium,Validate unique IDs,Pending


# Order Reviews Dataset

## Objective

The Order Reviews dataset captures customer satisfaction through review scores submitted after order completion.

Since customer experience is one of the primary business objectives of this project, ensuring the integrity of review data is essential.

The following quality checks will be performed:

- Dataset overview
- Primary key validation
- Duplicate validation
- Missing value analysis
- Review score validation
- Business rule validation
- Cleaning decisions
- Post-cleaning validation

In [58]:
# ==========================================================
# Order Reviews Dataset
# ==========================================================

reviews_df = cleaned_datasets["order_reviews_dataset"].copy()

print(f"Rows    : {reviews_df.shape[0]:,}")
print(f"Columns : {reviews_df.shape[1]}")

reviews_df.head()

Rows    : 99,224
Columns : 5


,review_id,order_id,review_score,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,2018-01-18,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,2018-03-10,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,2018-02-17,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,2017-04-21,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,2018-03-01,2018-03-02 10:26:53


In [59]:
# ==========================================================
# Reviews Dataset Summary
# ==========================================================

reviews_summary = pd.DataFrame({

    "Data Type": reviews_df.dtypes,

    "Missing Values": reviews_df.isna().sum(),

    "Missing %": (
        reviews_df.isna().mean()*100
    ).round(2),

    "Unique Values": reviews_df.nunique()

})

reviews_summary

,Data Type,Missing Values,Missing %,Unique Values
review_id,str,0,0.00,98410
order_id,str,0,0.00,98673
review_score,int64,0,0.00,5
review_creation_date,datetime64[us],0,0.00,636
review_answer_timestamp,datetime64[us],0,0.00,98248


In [60]:
duplicate_rows = reviews_df.duplicated().sum()

print(f"Duplicate Rows : {duplicate_rows}")

Duplicate Rows : 0


In [61]:
duplicate_review_ids = (
    reviews_df["review_id"]
    .duplicated()
    .sum()
)

print(
    f"Duplicate Review IDs : {duplicate_review_ids}"
)

Duplicate Review IDs : 814


In [62]:
duplicate_order_reviews = (
    reviews_df["order_id"]
    .duplicated()
    .sum()
)

print(
    f"Duplicate Order IDs : {duplicate_order_reviews}"
)

Duplicate Order IDs : 551


In [63]:
missing_percentage = (
    reviews_df
    .isna()
    .mean()
    .mul(100)
    .round(2)
)

missing_percentage.sort_values(
    ascending=False
)

review_id                 0.00
order_id                  0.00
review_score              0.00
review_creation_date      0.00
review_answer_timestamp   0.00
dtype: float64

In [64]:
reviews_df["review_score"].value_counts().sort_index()

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

In [65]:
invalid_scores = reviews_df[
    ~reviews_df["review_score"].between(1, 5)
]

print(f"Invalid Review Scores : {len(invalid_scores)}")

Invalid Review Scores : 0


In [66]:
reviews_df["order_id"].isna().sum()

np.int64(0)

In [67]:
reviews_cleaning = pd.DataFrame({

    "Issue": [
        "Duplicate Rows",
        "Duplicate Review IDs",
        "Duplicate Order IDs",
        "Missing Values",
        "Invalid Review Scores"
    ],

    "Observation": [
        "",
        "",
        "",
        "",
        ""
    ],

    "Decision": [
        "",
        "",
        "",
        "",
        ""
    ]
})

reviews_cleaning

,Issue,Observation,Decision
0,Duplicate Rows,,
1,Duplicate Review IDs,,
2,Duplicate Order IDs,,
3,Missing Values,,
4,Invalid Review Scores,,


## Duplicate Review ID Investigation

The initial assessment identified duplicate `review_id` values.

Before removing or modifying any records, duplicate review IDs will be inspected to determine whether they represent:

- Duplicate records
- Multiple orders sharing the same review
- Data quality issues
- Valid business behavior

In [68]:
# ==========================================================
# Duplicate Review IDs
# ==========================================================

duplicate_review_records = reviews_df[
    reviews_df["review_id"].duplicated(keep=False)
].sort_values("review_id")

print(f"Duplicate Review Records: {len(duplicate_review_records)}")

duplicate_review_records.head(20)

Duplicate Review Records: 1603


,review_id,order_id,review_score,review_creation_date,review_answer_timestamp
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,2018-03-07,2018-03-20 18:08:23
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,2018-03-07,2018-03-20 18:08:23
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,2017-09-21,2017-09-26 03:27:47
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,2017-09-21,2017-09-26 03:27:47
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,2018-03-07,2018-03-08 03:00:53
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,2018-03-07,2018-03-08 03:00:53
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,2018-03-02,2018-03-05 01:43:30
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,2018-03-02,2018-03-05 01:43:30
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,2017-09-09,2017-09-13 09:52:44
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,2017-09-09,2017-09-13 09:52:44


In [69]:
duplicate_review_summary = (
    duplicate_review_records
    .groupby("review_id")
    .size()
    .value_counts()
    .sort_index()
)

duplicate_review_summary

2    764
3     25
Name: count, dtype: int64

In [70]:
# ==========================================================
# Duplicate Order IDs
# ==========================================================

duplicate_order_records = reviews_df[
    reviews_df["order_id"].duplicated(keep=False)
].sort_values("order_id")

print(f"Duplicate Order Records: {len(duplicate_order_records)}")

duplicate_order_records.head(20)

Duplicate Order Records: 1098


,review_id,order_id,review_score,review_creation_date,review_answer_timestamp
25612,89a02c45c340aeeb1354a24e7d4b2c1e,0035246a40f520710769010f752e7507,5,2017-08-29,2017-08-30 01:59:12
22423,2a74b0559eb58fc1ff842ecc999594cb,0035246a40f520710769010f752e7507,5,2017-08-25,2017-08-29 21:45:57
22779,ab30810c29da5da8045216f0f62652a2,013056cfe49763c6f66bda03396c5ee3,5,2018-02-22,2018-02-23 12:12:30
68633,73413b847f63e02bc752b364f6d05ee9,013056cfe49763c6f66bda03396c5ee3,4,2018-03-04,2018-03-05 17:02:00
854,830636803620cdf8b6ffaf1b2f6e92b2,0176a6846bcb3b0d3aa3116a9a768597,5,2017-12-30,2018-01-02 10:54:06
83224,d8e8c42271c8fb67b9dad95d98c8ff80,0176a6846bcb3b0d3aa3116a9a768597,5,2017-12-30,2018-01-02 10:54:47
17582,017f0e1ea6386de662cbeba299c59ad1,02355020fd0a40a0d56df9f6ff060413,1,2018-03-29,2018-03-30 03:16:19
89888,0c8e7347f1cdd2aede37371543e3d163,02355020fd0a40a0d56df9f6ff060413,3,2018-03-21,2018-03-22 01:32:08
55137,61fe4e7d1ae801bbe169eb67b86c6eda,029863af4b968de1e5d6a82782e662f5,4,2017-07-19,2017-07-20 12:06:11
37911,04d945e95c788a3aa1ffbee42105637b,029863af4b968de1e5d6a82782e662f5,5,2017-07-14,2017-07-17 13:58:06


In [71]:
duplicate_review_records.duplicated().sum()

np.int64(0)

In [72]:
duplicate_review_records.sort_values(
    ["review_id", "order_id"]
).head(20)

,review_id,order_id,review_score,review_creation_date,review_answer_timestamp
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,2018-03-07,2018-03-20 18:08:23
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,2018-03-07,2018-03-20 18:08:23
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,2017-09-21,2017-09-26 03:27:47
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,2017-09-21,2017-09-26 03:27:47
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,2018-03-07,2018-03-08 03:00:53
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,2018-03-07,2018-03-08 03:00:53
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,2018-03-02,2018-03-05 01:43:30
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,2018-03-02,2018-03-05 01:43:30
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,2017-09-09,2017-09-13 09:52:44
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,2017-09-09,2017-09-13 09:52:44


In [73]:
review_distribution = pd.DataFrame({

    "Count": reviews_df["review_score"].value_counts().sort_index(),

    "Percentage": (
        reviews_df["review_score"]
        .value_counts(normalize=True)
        .sort_index()
        *100
    ).round(2)

})

review_distribution

,Count,Percentage
review_score,,
1,11424,11.51
2,3151,3.18
3,8179,8.24
4,19142,19.29
5,57328,57.78


In [74]:
reviews_validation = pd.DataFrame({

    "Validation Check":[

        "Duplicate Rows",
        "Duplicate Review IDs",
        "Duplicate Order IDs",
        "Missing Values",
        "Invalid Review Scores"

    ],

    "Result":[

        duplicate_rows,
        duplicate_review_ids,
        duplicate_order_reviews,
        reviews_df.isna().sum().sum(),
        len(invalid_scores)

    ],

    "Status":[

        "Pass",
        "Accepted",
        "Accepted",
        "Pass",
        "Pass"

    ]

})

reviews_validation

,Validation Check,Result,Status
0,Duplicate Rows,0,Pass
1,Duplicate Review IDs,814,Accepted
2,Duplicate Order IDs,551,Accepted
3,Missing Values,0,Pass
4,Invalid Review Scores,0,Pass


## Dataset Cleaning Summary

The Order Reviews dataset passed all structural validation checks.

Key findings:

- No duplicate records were found.
- No missing values exist.
- All review scores fall within the valid range of 1–5.
- Duplicate review IDs and duplicate order IDs were investigated and determined to represent valid business relationships rather than data quality issues.

No records were removed or modified during the cleaning process. The dataset is considered ready for feature engineering and exploratory data analysis.

# Order Payments Dataset

## Objective

The Order Payments dataset contains payment information associated with each order.

This dataset will later be used to calculate revenue, average order value, payment behavior, and customer spending patterns.

The following validations will be performed:

- Dataset overview
- Duplicate validation
- Missing value analysis
- Payment value validation
- Installment validation
- Payment method validation
- Multiple payment analysis
- Cleaning decisions
- Post-cleaning validation

In [75]:
# ==========================================================
# Order Payments Dataset
# ==========================================================

payments_df = cleaned_datasets["order_payments_dataset"].copy()

print(f"Rows    : {payments_df.shape[0]:,}")
print(f"Columns : {payments_df.shape[1]}")

payments_df.head()

Rows    : 103,886
Columns : 5


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [76]:
payments_summary = pd.DataFrame({

    "Data Type": payments_df.dtypes,

    "Missing Values": payments_df.isna().sum(),

    "Missing %": (
        payments_df.isna().mean()*100
    ).round(2),

    "Unique Values": payments_df.nunique()

})

payments_summary

,Data Type,Missing Values,Missing %,Unique Values
order_id,str,0,0.00,99440
payment_sequential,int64,0,0.00,29
payment_type,str,0,0.00,5
payment_installments,int64,0,0.00,24
payment_value,float64,0,0.00,29077


In [77]:
duplicate_rows = payments_df.duplicated().sum()

print(f"Duplicate Rows : {duplicate_rows}")

Duplicate Rows : 0


In [78]:
duplicate_payment_records = payments_df.duplicated(

    subset=[
        "order_id",
        "payment_sequential"
    ]

).sum()

print(
    f"Duplicate Payment Records : {duplicate_payment_records}"
)

Duplicate Payment Records : 0


In [79]:
missing_percentage = (

    payments_df
    .isna()
    .mean()
    .mul(100)
    .round(2)

)

missing_percentage.sort_values(
    ascending=False
)

order_id               0.00
payment_sequential     0.00
payment_type           0.00
payment_installments   0.00
payment_value          0.00
dtype: float64

In [80]:
payments_df["payment_type"].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [81]:
invalid_payment_value = payments_df[
    payments_df["payment_value"] <= 0
]

print(
    f"Invalid Payment Values : {len(invalid_payment_value)}"
)

invalid_payment_value.head()

Invalid Payment Values : 9


,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.00
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.00
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.00
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.00
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.00


In [82]:
invalid_installments = payments_df[
    payments_df["payment_installments"] <= 0
]

print(
    f"Invalid Installments : {len(invalid_installments)}"
)

invalid_installments.head()

Invalid Installments : 2


,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


In [83]:
multiple_payments = (

    payments_df
    .groupby("order_id")
    .size()

)

multiple_payments.value_counts().sort_index()

1     96479
2      2382
3       301
4       108
5        52
6        36
7        28
8        11
9         9
10        5
11        8
12        8
13        3
14        2
15        2
19        2
21        1
22        1
26        1
29        1
Name: count, dtype: int64

In [84]:
multiple_payment_orders = payments_df[

    payments_df["order_id"].isin(

        multiple_payments[
            multiple_payments > 1
        ].index

    )

].sort_values(

    ["order_id","payment_sequential"]

)

multiple_payment_orders.head(20)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
89575,0016dfedd97fc2950e388d2971d718c7,1,credit_card,5,52.63
80856,0016dfedd97fc2950e388d2971d718c7,2,voucher,1,17.92
20036,002f19a65a2ddd70a090297872e6d64e,1,voucher,1,44.11
98894,002f19a65a2ddd70a090297872e6d64e,2,voucher,1,33.18
10244,0071ee2429bc1efdc43aa3e073a5290e,1,voucher,1,100.00
30155,0071ee2429bc1efdc43aa3e073a5290e,2,voucher,1,92.44
16053,009ac365164f8e06f59d18a08045f6c4,1,credit_card,1,0.88
16459,009ac365164f8e06f59d18a08045f6c4,2,voucher,1,4.50
73837,009ac365164f8e06f59d18a08045f6c4,3,voucher,1,8.25
32058,009ac365164f8e06f59d18a08045f6c4,4,voucher,1,5.45


In [85]:
payments_cleaning = pd.DataFrame({

    "Issue":[

        "Duplicate Rows",
        "Duplicate Payment Records",
        "Missing Values",
        "Invalid Payment Values",
        "Invalid Installments",
        "Multiple Payments"

    ],

    "Observation":[
        "",
        "",
        "",
        "",
        "",
        ""
    ],

    "Decision":[
        "",
        "",
        "",
        "",
        "",
        ""
    ]

})

payments_cleaning

,Issue,Observation,Decision
0,Duplicate Rows,,
1,Duplicate Payment Records,,
2,Missing Values,,
3,Invalid Payment Values,,
4,Invalid Installments,,
5,Multiple Payments,,


In [86]:
payments_validation = pd.DataFrame({

    "Validation Check":[

        "Duplicate Rows",
        "Duplicate Payment Records",
        "Missing Values",
        "Invalid Payment Values",
        "Invalid Installments"

    ],

    "Result":[

        duplicate_rows,
        duplicate_payment_records,
        payments_df.isna().sum().sum(),
        len(invalid_payment_value),
        len(invalid_installments)

    ],

    "Status":[

        "",
        "",
        "",
        "",
        ""

    ]

})

payments_validation

,Validation Check,Result,Status
0,Duplicate Rows,0,
1,Duplicate Payment Records,0,
2,Missing Values,0,
3,Invalid Payment Values,9,
4,Invalid Installments,2,


## Dataset Cleaning Summary

The Order Payments dataset was validated for structural integrity, payment consistency, and business rules.

Key validations included:

- Duplicate transaction detection
- Missing value analysis
- Payment amount validation
- Installment validation
- Payment method verification
- Investigation of multiple payment records per order

Only confirmed data quality issues, if any, will be corrected. Valid business scenarios such as split payments across multiple payment methods will be retained.

The cleaned dataset is now ready for feature engineering and exploratory data analysis.

In [87]:
payments_cleaning = pd.DataFrame({

    "Issue":[
        "Duplicate Rows",
        "Duplicate Payment Records",
        "Missing Values",
        "Invalid Payment Values",
        "Invalid Installments",
        "Multiple Payments"
    ],

    "Observation":[
        "No duplicate rows found",
        "No duplicate payment records found",
        "No missing values",
        "9 payment records have zero payment value",
        "2 payment records have zero installments",
        "Multiple payments per order represent valid split-payment transactions"
    ],

    "Decision":[
        "No action required",
        "No action required",
        "No action required",
        "Retain and document as business anomalies",
        "Retain and document as rare anomalies",
        "Retain for analysis"
    ]

})

payments_cleaning

,Issue,Observation,Decision
0,Duplicate Rows,No duplicate rows found,No action required
1,Duplicate Payment Records,No duplicate payment records found,No action required
2,Missing Values,No missing values,No action required
3,Invalid Payment Values,9 payment records have zero payment value,Retain and document as business anomalies
4,Invalid Installments,2 payment records have zero installments,Retain and document as rare anomalies
5,Multiple Payments,Multiple payments per order represent valid split-payment transactions,Retain for analysis


In [88]:
payments_validation = pd.DataFrame({

    "Validation Check":[
        "Duplicate Rows",
        "Duplicate Payment Records",
        "Missing Values",
        "Invalid Payment Values",
        "Invalid Installments"
    ],

    "Result":[
        duplicate_rows,
        duplicate_payment_records,
        payments_df.isna().sum().sum(),
        len(invalid_payment_value),
        len(invalid_installments)
    ],

    "Status":[
        "Pass",
        "Pass",
        "Pass",
        "Accepted",
        "Accepted"
    ]

})

payments_validation

,Validation Check,Result,Status
0,Duplicate Rows,0,Pass
1,Duplicate Payment Records,0,Pass
2,Missing Values,0,Pass
3,Invalid Payment Values,9,Accepted
4,Invalid Installments,2,Accepted


# 6. Order Items Dataset Validation

In [89]:
# ============================================================
# Order Items Dataset Overview
# ============================================================

order_items_df = datasets["order_items_dataset"].copy()

print(f"Rows    : {len(order_items_df):,}")
print(f"Columns : {order_items_df.shape[1]}")

display(order_items_df.head())

Rows    : 112,650
Columns : 7


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [90]:
# ============================================================
# Dataset Information
# ============================================================

summary = pd.DataFrame({
    "Data Type": order_items_df.dtypes.astype(str),
    "Missing Values": order_items_df.isnull().sum(),
    "Missing %": round(order_items_df.isnull().mean()*100,2),
    "Unique Values": order_items_df.nunique()
})

display(summary)

,Data Type,Missing Values,Missing %,Unique Values
order_id,str,0,0.00,98666
order_item_id,int64,0,0.00,21
product_id,str,0,0.00,32951
seller_id,str,0,0.00,3095
shipping_limit_date,datetime64[us],0,0.00,93318
price,float64,0,0.00,5968
freight_value,float64,0,0.00,6999


In [91]:
# ============================================================
# Duplicate Validation
# ============================================================

duplicate_rows = order_items_df.duplicated().sum()

print("Duplicate Rows :", duplicate_rows)

duplicate_order_items = order_items_df.duplicated(
    subset=["order_id","order_item_id"]
).sum()

print("Duplicate Order-Item Keys :", duplicate_order_items)

Duplicate Rows : 0
Duplicate Order-Item Keys : 0


In [92]:
# ============================================================
# Missing Value Analysis
# ============================================================

missing_percent = (
    order_items_df.isnull()
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
)

display(missing_percent)

order_id              0.00
order_item_id         0.00
product_id            0.00
seller_id             0.00
shipping_limit_date   0.00
price                 0.00
freight_value         0.00
dtype: float64

In [93]:
# ============================================================
# Primary Key Validation
# ============================================================

duplicate_keys = order_items_df[
    order_items_df.duplicated(
        subset=["order_id","order_item_id"],
        keep=False
    )
]

print("Duplicate Composite Keys :", len(duplicate_keys))

display(duplicate_keys.head(20))

Duplicate Composite Keys : 0


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value


In [94]:
# ============================================================
# Product & Seller Validation
# ============================================================

print("Missing Product IDs :",
      order_items_df["product_id"].isna().sum())

print("Missing Seller IDs :",
      order_items_df["seller_id"].isna().sum())

Missing Product IDs : 0
Missing Seller IDs : 0


In [95]:
# ============================================================
# Numeric Validation
# ============================================================

invalid_price = order_items_df[
    order_items_df["price"] <= 0
]

print("Invalid Prices :", len(invalid_price))

display(invalid_price.head())

invalid_freight = order_items_df[
    order_items_df["freight_value"] < 0
]

print("Negative Freight :", len(invalid_freight))

display(invalid_freight.head())

invalid_item = order_items_df[
    order_items_df["order_item_id"] <= 0
]

print("Invalid Order Item IDs :", len(invalid_item))

display(invalid_item.head())

Invalid Prices : 0


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value


Negative Freight : 0


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value


Invalid Order Item IDs : 0


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value


In [96]:
# ============================================================
# Shipping Date Validation
# ============================================================

order_items_df["shipping_limit_date"] = pd.to_datetime(
    order_items_df["shipping_limit_date"],
    errors="coerce"
)

print("Invalid Shipping Dates :",
      order_items_df["shipping_limit_date"].isna().sum())

print(order_items_df["shipping_limit_date"].dtype)

Invalid Shipping Dates : 0
datetime64[us]


In [97]:
# ============================================================
# Price Statistics
# ============================================================

display(order_items_df["price"].describe())

display(order_items_df["freight_value"].describe())

count   112650.00
mean       120.65
std        183.63
min          0.85
25%         39.90
50%         74.99
75%        134.90
max       6735.00
Name: price, dtype: float64

count   112650.00
mean        19.99
std         15.81
min          0.00
25%         13.08
50%         16.26
75%         21.15
max        409.68
Name: freight_value, dtype: float64

In [98]:
# ============================================================
# Business Validation
# ============================================================

zero_freight = order_items_df[
    order_items_df["freight_value"] == 0
]

print("Free Shipping Orders :", len(zero_freight))

display(zero_freight.head())

high_price = order_items_df[
    order_items_df["price"] >
    order_items_df["price"].quantile(0.99)
]

print("Top 1% Expensive Products :", len(high_price))

display(high_price.head())

high_freight = order_items_df[
    order_items_df["freight_value"] >
    order_items_df["freight_value"].quantile(0.99)
]

print("Top 1% Freight Charges :", len(high_freight))

display(high_freight.head())

Free Shipping Orders : 383


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
114,00404fa7a687c8c44ca69d42695aae73,1,53b36df67ebb7c41585e8d54d6772e08,7d13fca15225358621be4086e1eb0964,2018-05-15 04:31:26,99.90,0.00
258,00a870c6c06346e85335524935c600c0,1,aca2eb7d00ea1a7b8ebd4e68314663af,955fee9216a65b617aa5c0531780ce60,2018-05-14 00:14:29,69.90,0.00
483,011c899816ea29773525bd3322dbb6aa,1,53b36df67ebb7c41585e8d54d6772e08,7d13fca15225358621be4086e1eb0964,2018-05-07 05:30:45,99.90,0.00
508,012b3f6ab7776a8ab3443a4ad7bef2e6,1,422879e10f46682990de24d770e7f83d,1f50f920176fa81dab994f9023523100,2018-05-09 21:30:50,53.90,0.00
509,012b3f6ab7776a8ab3443a4ad7bef2e6,2,422879e10f46682990de24d770e7f83d,1f50f920176fa81dab994f9023523100,2018-05-09 21:30:50,53.90,0.00


Top 1% Expensive Products : 1117


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
322,00c9474e0334f7a4ffc8c3a8bd21a51e,1,4b2653088591de362e6ba85b4a474c75,610f72e407cdd7caaa2f8167b0163fd8,2018-05-10 15:50:46,1050.61,25.13
475,011a43bc9bb525517251ebb3ebc99b69,1,ca7966fa77959536468be3c6ce1f19e1,610f72e407cdd7caaa2f8167b0163fd8,2018-05-03 18:09:38,899.00,21.20
518,012f2c4ca09b101a73e18957c3294cd6,1,a9e9edb1bcac585bfbfa381ce40e5d99,06532f10282704ef4c69168b914b77be,2017-05-11 22:35:11,1820.00,81.62
865,02014f2495eef0e869616829d481d743,1,43cc8e4d981bc04b9d78b12e8a908d41,6061155addc1e54b4cfb51c1c2a32ad8,2018-08-24 11:05:25,1240.00,102.63
873,02058d1c3b825765a4ec45968b8a1c97,1,d05d4a7430d1293367ba2ffbedcdef05,54065e9aef7e9e9c2dc23b7594db021a,2017-06-02 02:50:28,1799.00,27.34


Top 1% Freight Charges : 1124


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
124,00471463a6106056c1a2a809f70de640,1,9df0e8a7eef2a38b74e6d5c0e224b11f,7c67e1448b00f6e969d365cea6b010ab,2017-10-03 22:46:14,179.99,85.97
162,0066a1fdaee16ad5022c5ef979d0b661,1,2fb0efd1f61f186ffdda9e8ec70f27f2,1d8dbc4f32378d715c717c1c1fc57bae,2018-06-14 20:17:26,139.00,87.28
352,00d88c00221ce96a1cfcaca9f2fceb3b,1,7814c273ab16783d73a9863ebfa8b141,1025f0e2d44d7041d6cf58b6550e0bfa,2018-03-15 23:15:29,250.00,97.56
456,011108c8a3d6eee6807f48a2e639439f,1,ea11e700a343582ad56e4c70e966cb36,01cf7e3d21494c41fb86034f2e714fa1,2017-05-02 17:42:48,593.36,98.13
590,015fb6b5f739788434fa690540f90f19,1,a0eccc912d428eaddf1ef4fe47593ece,8a87611c08849ffeeccab52aa798b6c7,2017-02-04 13:06:05,143.00,94.98


In [99]:
# ============================================================
# Cleaning Decisions
# ============================================================

order_items_cleaning = pd.DataFrame({

    "Issue":[
        "Duplicate Rows",
        "Duplicate Order-Item Keys",
        "Missing Product IDs",
        "Missing Seller IDs",
        "Invalid Prices",
        "Negative Freight",
        "Invalid Item IDs",
        "Shipping Date Validation"
    ],

    "Observation":[
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        ""
    ],

    "Decision":[
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        ""
    ]

})

display(order_items_cleaning)

,Issue,Observation,Decision
0,Duplicate Rows,,
1,Duplicate Order-Item Keys,,
2,Missing Product IDs,,
3,Missing Seller IDs,,
4,Invalid Prices,,
5,Negative Freight,,
6,Invalid Item IDs,,
7,Shipping Date Validation,,


In [100]:
# ============================================================
# Validation Summary
# ============================================================

validation = pd.DataFrame({

    "Validation Check":[
        "Duplicate Rows",
        "Duplicate Order-Item Keys",
        "Missing Product IDs",
        "Missing Seller IDs",
        "Invalid Prices",
        "Negative Freight",
        "Invalid Item IDs"
    ],

    "Result":[
        duplicate_rows,
        duplicate_order_items,
        order_items_df["product_id"].isna().sum(),
        order_items_df["seller_id"].isna().sum(),
        len(invalid_price),
        len(invalid_freight),
        len(invalid_item)
    ],

    "Status":[
        "",
        "",
        "",
        "",
        "",
        "",
        ""
    ]

})

display(validation)

,Validation Check,Result,Status
0,Duplicate Rows,0,
1,Duplicate Order-Item Keys,0,
2,Missing Product IDs,0,
3,Missing Seller IDs,0,
4,Invalid Prices,0,
5,Negative Freight,0,
6,Invalid Item IDs,0,


In [101]:
order_items_cleaning = pd.DataFrame({

    "Issue":[
        "Duplicate Rows",
        "Duplicate Order-Item Keys",
        "Missing Product IDs",
        "Missing Seller IDs",
        "Invalid Prices",
        "Negative Freight",
        "Invalid Item IDs",
        "Shipping Date Validation"
    ],

    "Observation":[
        "No duplicate rows found",
        "Composite key (order_id, order_item_id) is unique",
        "No missing Product IDs",
        "No missing Seller IDs",
        "All prices are positive",
        "No negative freight charges",
        "All order item IDs are valid",
        "All shipping dates are valid datetime values"
    ],

    "Decision":[
        "No action required",
        "No action required",
        "No action required",
        "No action required",
        "No action required",
        "No action required",
        "No action required",
        "No action required"
    ]

})

display(order_items_cleaning)

,Issue,Observation,Decision
0,Duplicate Rows,No duplicate rows found,No action required
1,Duplicate Order-Item Keys,"Composite key (order_id, order_item_id) is unique",No action required
2,Missing Product IDs,No missing Product IDs,No action required
3,Missing Seller IDs,No missing Seller IDs,No action required
4,Invalid Prices,All prices are positive,No action required
5,Negative Freight,No negative freight charges,No action required
6,Invalid Item IDs,All order item IDs are valid,No action required
7,Shipping Date Validation,All shipping dates are valid datetime values,No action required


In [102]:
validation = pd.DataFrame({

    "Validation Check":[
        "Duplicate Rows",
        "Duplicate Order-Item Keys",
        "Missing Product IDs",
        "Missing Seller IDs",
        "Invalid Prices",
        "Negative Freight",
        "Invalid Item IDs",
        "Invalid Shipping Dates"
    ],

    "Result":[
        duplicate_rows,
        duplicate_order_items,
        order_items_df["product_id"].isna().sum(),
        order_items_df["seller_id"].isna().sum(),
        len(invalid_price),
        len(invalid_freight),
        len(invalid_item),
        order_items_df["shipping_limit_date"].isna().sum()
    ],

    "Status":[
        "Pass",
        "Pass",
        "Pass",
        "Pass",
        "Pass",
        "Pass",
        "Pass",
        "Pass"
    ]

})

display(validation)

,Validation Check,Result,Status
0,Duplicate Rows,0,Pass
1,Duplicate Order-Item Keys,0,Pass
2,Missing Product IDs,0,Pass
3,Missing Seller IDs,0,Pass
4,Invalid Prices,0,Pass
5,Negative Freight,0,Pass
6,Invalid Item IDs,0,Pass
7,Invalid Shipping Dates,0,Pass


# 7. Customers Dataset Validation

In [103]:
# ============================================================
# Customers Dataset Overview
# ============================================================

customers_df = datasets["customers_dataset"].copy()

print(f"Rows    : {len(customers_df):,}")
print(f"Columns : {customers_df.shape[1]}")

display(customers_df.head())

Rows    : 99,441
Columns : 5


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [104]:
# ============================================================
# Dataset Information
# ============================================================

summary = pd.DataFrame({

    "Data Type": customers_df.dtypes.astype(str),
    "Missing Values": customers_df.isnull().sum(),
    "Missing %": round(customers_df.isnull().mean()*100,2),
    "Unique Values": customers_df.nunique()

})

display(summary)

,Data Type,Missing Values,Missing %,Unique Values
customer_id,str,0,0.00,99441
customer_unique_id,str,0,0.00,96096
customer_zip_code_prefix,int64,0,0.00,14994
customer_city,str,0,0.00,4119
customer_state,str,0,0.00,27


In [105]:
# ============================================================
# Duplicate Validation
# ============================================================

duplicate_rows = customers_df.duplicated().sum()

print("Duplicate Rows :", duplicate_rows)

duplicate_customer_id = customers_df["customer_id"].duplicated().sum()

print("Duplicate Customer IDs :", duplicate_customer_id)

duplicate_customer_unique = customers_df["customer_unique_id"].duplicated().sum()

print("Duplicate Customer Unique IDs :", duplicate_customer_unique)

Duplicate Rows : 0
Duplicate Customer IDs : 0
Duplicate Customer Unique IDs : 3345


In [106]:
# ============================================================
# Missing Values
# ============================================================

missing_percent = (
    customers_df.isnull()
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
)

display(missing_percent)

customer_id                0.00
customer_unique_id         0.00
customer_zip_code_prefix   0.00
customer_city              0.00
customer_state             0.00
dtype: float64

In [107]:
# ============================================================
# ZIP Code Validation
# ============================================================

invalid_zip = customers_df[
    customers_df["customer_zip_code_prefix"] <= 0
]

print("Invalid ZIP Codes :", len(invalid_zip))

display(invalid_zip.head())

Invalid ZIP Codes : 0


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state


In [108]:
# ============================================================
# City & State Validation
# ============================================================

print("Missing Cities :",
      customers_df["customer_city"].isna().sum())

print("Missing States :",
      customers_df["customer_state"].isna().sum())

print()

print(customers_df["customer_state"].value_counts())

Missing Cities : 0
Missing States : 0

customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
PE     1652
CE     1336
PA      975
MT      907
MA      747
MS      715
PB      536
PI      495
RN      485
AL      413
SE      350
TO      280
RO      253
AM      148
AC       81
AP       68
RR       46
Name: count, dtype: int64


In [109]:
# ============================================================
# Business Validation
# ============================================================

customers_per_person = (
    customers_df.groupby("customer_unique_id")
    .size()
)

print(customers_per_person.describe())

print()

print("Customers with multiple accounts :",
      (customers_per_person > 1).sum())

display(customers_per_person.sort_values(ascending=False).head(20))

count   96096.00
mean        1.03
std         0.21
min         1.00
25%         1.00
50%         1.00
75%         1.00
max        17.00
dtype: float64

Customers with multiple accounts : 2997


customer_unique_id
8d50f5eadf50201ccdcedfb9e2ac8455    17
3e43e6105506432c953e165fb2acf44c     9
6469f99c1f9dfae7733b25662e7f1782     7
ca77025e7201e3b30c44b472ff346268     7
1b6c7548a2a1f9037c1fd3ddfed95f33     7
12f5d6e1cbf93dafd9dcc19095df0b3d     6
de34b16117594161a6a89c50b289d35a     6
63cfc61cee11cbe306bff5857d00bfe4     6
f0e310a6839dce9de1638e0fe5ab282a     6
47c1a3033b8b77b3ab6e109eb4d5fdf3     6
dc813062e0fc23409cd255f7f53c7074     6
4e65032f1f574189fb793bac5a867bbc     5
b4e4f24de1e8725b74e4a1f4975116ed     5
394ac4de8f3acb14253c177f0e15bc58     5
5e8f38a9a1c023f3db718edcf926a2db     5
35ecdf6858edc6427223b64804cf028e     5
74cb1ad7e6d5674325c1f99b5ea30d82     5
fe81bb32c243a86b2f86fbf053fe6140     5
56c8638e7c058b98aae6d74d2dd6ea23     5
795c1622cf7a53d63d324e862349d01c     4
dtype: int64

In [110]:
# ============================================================
# Cleaning Decisions
# ============================================================

customers_cleaning = pd.DataFrame({

    "Issue":[
        "Duplicate Rows",
        "Duplicate Customer IDs",
        "Duplicate Customer Unique IDs",
        "Missing Values",
        "Invalid ZIP Codes",
        "Customer-State Validation"
    ],

    "Observation":[
        "",
        "",
        "",
        "",
        "",
        ""
    ],

    "Decision":[
        "",
        "",
        "",
        "",
        "",
        ""
    ]

})

display(customers_cleaning)

,Issue,Observation,Decision
0,Duplicate Rows,,
1,Duplicate Customer IDs,,
2,Duplicate Customer Unique IDs,,
3,Missing Values,,
4,Invalid ZIP Codes,,
5,Customer-State Validation,,


In [111]:
# ============================================================
# Validation Summary
# ============================================================

validation = pd.DataFrame({

    "Validation Check":[
        "Duplicate Rows",
        "Duplicate Customer IDs",
        "Missing Values",
        "Invalid ZIP Codes",
        "Customer-State Validation"
    ],

    "Result":[
        duplicate_rows,
        duplicate_customer_id,
        customers_df.isna().sum().sum(),
        len(invalid_zip),
        customers_df["customer_state"].isna().sum()
    ],

    "Status":[
        "",
        "",
        "",
        "",
        ""
    ]

})

display(validation)

,Validation Check,Result,Status
0,Duplicate Rows,0,
1,Duplicate Customer IDs,0,
2,Missing Values,0,
3,Invalid ZIP Codes,0,
4,Customer-State Validation,0,


Geolocation Dataset Validation Checklist

In [123]:
geo = datasets["geolocation_dataset"].copy()

print("Rows    :", geo.shape[0])
print("Columns :", geo.shape[1])

geo.head()

Rows    : 1000163
Columns : 5


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.55,-46.64,sao paulo,SP
1,1046,-23.55,-46.64,sao paulo,SP
2,1046,-23.55,-46.64,sao paulo,SP
3,1041,-23.54,-46.64,sao paulo,SP
4,1035,-23.54,-46.64,sao paulo,SP


In [124]:
summary = pd.DataFrame({
    "Data Type": geo.dtypes.astype(str),
    "Missing Values": geo.isnull().sum(),
    "Missing %": round((geo.isnull().sum()/len(geo))*100,2),
    "Unique Values": geo.nunique()
})

summary

,Data Type,Missing Values,Missing %,Unique Values
geolocation_zip_code_prefix,int64,0,0.00,19015
geolocation_lat,float64,0,0.00,717372
geolocation_lng,float64,0,0.00,717615
geolocation_city,str,0,0.00,8011
geolocation_state,str,0,0.00,27


In [125]:
duplicate_rows = geo.duplicated().sum()

print("Duplicate Rows :", duplicate_rows)

Duplicate Rows : 261831


In [126]:
duplicate_zip = geo["geolocation_zip_code_prefix"].duplicated().sum()

print("Duplicate ZIP Prefixes :", duplicate_zip)

Duplicate ZIP Prefixes : 981148


In [127]:
geo.isnull().mean()*100

geolocation_zip_code_prefix   0.00
geolocation_lat               0.00
geolocation_lng               0.00
geolocation_city              0.00
geolocation_state             0.00
dtype: float64

In [128]:
invalid_lat = geo[
    (geo["geolocation_lat"] < -34) |
    (geo["geolocation_lat"] > 6)
]

print("Invalid Latitudes :", len(invalid_lat))

invalid_lat.head()

Invalid Latitudes : 31


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
387565,18243,28.01,-15.54,bom retiro da esperanca,SP
513631,28165,41.61,-8.41,vila nova de campos,RJ
513643,28155,-34.59,-58.73,santa maria,RJ
513754,28155,42.44,13.82,santa maria,RJ
514429,28333,38.38,-6.33,raposo,RJ


In [129]:
invalid_lon = geo[
    (geo["geolocation_lng"] < -75) |
    (geo["geolocation_lng"] > -30)
]

print("Invalid Longitudes :", len(invalid_lon))

invalid_lon.head()

Invalid Longitudes : 26


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
387565,18243,28.01,-15.54,bom retiro da esperanca,SP
513631,28165,41.61,-8.41,vila nova de campos,RJ
513754,28155,42.44,13.82,santa maria,RJ
514429,28333,38.38,-6.33,raposo,RJ
516682,28595,43.68,-7.41,portela,RJ


In [130]:
missing_city = geo["geolocation_city"].isna().sum()

print("Missing Cities :", missing_city)

Missing Cities : 0


In [131]:
missing_state = geo["geolocation_state"].isna().sum()

print("Missing States :", missing_state)

Missing States : 0


In [132]:
geo["geolocation_state"].value_counts()

geolocation_state
SP    404268
MG    126336
RJ    121169
RS     61851
PR     57859
SC     38328
BA     36045
GO     20139
ES     16748
PE     16432
DF     12986
MT     12031
CE     11674
PA     10853
MS     10431
MA      7853
PB      5538
RN      5041
PI      4549
AL      4183
TO      3576
SE      3563
RO      3478
AM      2432
AC      1301
AP       853
RR       646
Name: count, dtype: int64

In [133]:
duplicate_coordinates = geo.duplicated(
    subset=["geolocation_lat","geolocation_lng"]
).sum()

print("Duplicate Coordinates :", duplicate_coordinates)

Duplicate Coordinates : 281700
